# Dziennik projektu EEG

**Projekt:** BIAI / EEG Based Visual Recall  
**Cel roboczy:** sprawdzić, czy z sygnału EEG da się przewidywać kategorię obrazu, a później iść w stronę rekonstrukcji / embeddingów obrazów.  
**Data utworzenia notebooka:** 2026-06-11

Ten notebook jest dokumentacją prac wykonanych do tej pory. Przy kolejnych zmianach będę dopisywał tutaj nową sekcję w dzienniku zmian oraz aktualizował wyniki.

## 1. Dane wejściowe

Główne pliki danych używane w obecnym pipeline:

- `dane/mole_/mole_0004_raw.edf` - główny zapis EEG z triggerami.
- `dane/mole_/mole_0004_imp.csv` - jakość kanałów / impedancje.
- `dane/mole_/mole_EEGBasedVisualRecall_Events_Rep2_2026-05-22_11-27-28.csv` - metadane eventów: trial, obraz, kategoria, trigger.

Najważniejsze triggery:

| Kod | Nazwa | Znaczenie |
|---:|---|---|
| 10 | `FIXATION` | punkt fiksacji |
| 11 | `BLACK_SCREEN_PRE_IMAGE` | czarny ekran przed obrazem |
| 12 | `IMAGE_ON` | pokazanie obrazu |
| 13 | `BLACK_SCREEN_POST_IMAGE` | czarny ekran po obrazie |
| 14 | `DESCRIBE_SCREEN` | ekran opisu / przypominania |
| 15 | `SPACE_PRESSED` | zakończenie opisu |
| 16 | `BLACK_SCREEN_POST_SPACE` | czarny ekran po spacji |

Obecnie głównym triggerem eksperymentalnym jest `12 = IMAGE_ON`.

## 2. Pipeline przygotowania danych

Obecnie są dwa główne tory przygotowania danych.

### Tor A: spektrogramy

Plik: `build_event_spectrogram_dataset.py`

Algorytm:

1. Wczytanie EDF przez `mne`.
2. Czyszczenie nazw kanałów, np. `EEG Fp1-Pz` -> `Fp1`.
3. Oznaczenie kanałów technicznych jako `misc` / `stim`.
4. Montaż `standard_1020`.
5. Wykrycie złych kanałów z pliku impedancji (`Low_SNR > 0.5`).
6. Interpolacja złych kanałów.
7. Filtr pasmowy `1-40 Hz`.
8. Notch `50 Hz`.
9. Common Average Reference.
10. Mapowanie adnotacji EDF do eventów.
11. Dołączenie metadanych z CSV: `trial_id`, `image_id`, `image_file`, `image_category`.
12. Wycięcie epoki wokół triggera `IMAGE_ON`.
13. STFT / `scipy.signal.spectrogram`.
14. Zapis tensora `.npz` i podglądu `.png`.

Domyślne okno:

```text
tmin = -0.5 s
tmax =  1.5 s
```

Wynikowy tensor spektrogramu:

```text
channels x freqs x times = 21 x 17 x 30
```

### Tor B: raw epoki EEG

Plik: `build_event_epoch_dataset.py`

Ten tor robi ten sam preprocessing EEG, ale nie liczy spektrogramu. Zapisuje surową epokę EEG wokół triggera.

Wynikowy tensor raw epoch:

```text
channels x samples = 21 x 1200
```

Przy próbkowaniu `600 Hz` oznacza to 2 sekundy sygnału: `-0.5s .. +1.5s` względem pokazania obrazu.

## 3. Wygenerowane datasety

### Spektrogramy

Folder:

```text
event_spectrogram_dataset/
```

Zawartość:

- `metadata.csv` - metadane próbek.
- `config.json` - parametry generowania.
- `tensors/*.npz` - tensory spektrogramów.
- `previews/*.png` - wizualne podglądy.

Liczba próbek: `1980`.

### Raw epoki

Folder:

```text
event_epoch_dataset/
```

Zawartość:

- `metadata.csv` - metadane próbek.
- `config.json` - parametry generowania.
- `epochs/*.npz` - raw epoki EEG.

Liczba próbek: `1980`.

## 4. Modele i wyniki

Wszystkie wyniki poniżej są liczone na podziale grupowanym po `series_id`. To znaczy, że test zawiera serie niewidziane podczas treningu. Aktualny test to serie `[2, 8]`.

| Model | Wejście | Wynik accuracy |
|---|---|---:|
| Losowy baseline | etykiety kategorii | 9.09% |
| LogisticRegression | spłaszczony spektrogram `21 x 17 x 30` | 22.73% |
| Mały CNN | spektrogram `21 x 17 x 30` | 20.45% |
| EEGNet | raw epoka `1 x 21 x 1200` | **26.36%** |

Aktualnie najlepszy jest **EEGNet na surowych epokach EEG**.

Interpretacja robocza:

- Wyniki są wyraźnie powyżej losowego `9.09%`, więc sygnał EEG niesie jakąś informację o kategorii obrazu.
- CNN na spektrogramie nie przebił prostego modelu liniowego, więc obecna reprezentacja spektrogramu może tracić część informacji.
- EEGNet działa najlepiej, co sugeruje, że warto dalej badać surowe epoki czasowe.

## 5. Pliki kodu dodane do projektu

| Plik | Rola |
|---|---|
| `build_event_spectrogram_dataset.py` | generuje spektrogramy z triggerów |
| `train_spectrogram_baseline.py` | baseline LogisticRegression na spektrogramach |
| `train_spectrogram_cnn.py` | mały CNN w PyTorch na spektrogramach |
| `build_event_epoch_dataset.py` | generuje raw epoki EEG |
| `train_eegnet.py` | EEGNet na raw epokach |
| `PLAN_SPEKTROGRAMY_I_SIECI.md` | plan i skrócone wyniki eksperymentów |

Dodatkowo utworzone zostało lokalne środowisko Python:

```text
.venv313/
```

W nim jest PyTorch i zależności do treningu modeli.

## 6. Komendy odtwarzające eksperymenty

### Generowanie spektrogramów

```powershell
python .\build_event_spectrogram_dataset.py --output-dir event_spectrogram_dataset
```

### Baseline LogisticRegression

```powershell
python .\train_spectrogram_baseline.py --dataset-dir event_spectrogram_dataset --output-dir baseline_results --split series
```

### CNN na spektrogramach

```powershell
.\.venv313\Scripts\python.exe .\train_spectrogram_cnn.py --dataset-dir event_spectrogram_dataset --output-dir cnn_results --split series --epochs 35 --batch-size 64 --cpu
```

### Generowanie raw epok

```powershell
python .\build_event_epoch_dataset.py --output-dir event_epoch_dataset
```

### EEGNet

```powershell
.\.venv313\Scripts\python.exe .\train_eegnet.py --dataset-dir event_epoch_dataset --output-dir eegnet_results --split series --epochs 20 --batch-size 128 --cpu
```

## 7. Wyniki zapisane na dysku

### Baseline

```text
baseline_results/spectrogram_baseline_summary.txt
baseline_results/spectrogram_baseline_confusion_matrix.csv
```

### CNN

```text
cnn_results/spectrogram_cnn.pt
cnn_results/spectrogram_cnn_report.txt
cnn_results/spectrogram_cnn_history.csv
cnn_results/spectrogram_cnn_confusion_matrix.csv
cnn_results/spectrogram_cnn_summary.json
```

### EEGNet

```text
eegnet_results/eegnet.pt
eegnet_results/eegnet_report.txt
eegnet_results/eegnet_history.csv
eegnet_results/eegnet_confusion_matrix.csv
eegnet_results/eegnet_summary.json
```

## 8. Co robić dalej

Najlepszy następny krok to porównać różne okna czasowe i różne triggery dla EEGNet.

Proponowane warianty:

```powershell
python .\build_event_epoch_dataset.py --tmin 0.0 --tmax 0.8 --output-dir event_epoch_dataset_image_on_early
python .\build_event_epoch_dataset.py --tmin -0.2 --tmax 1.0 --output-dir event_epoch_dataset_image_on_mid
python .\build_event_epoch_dataset.py --event-code 14 --tmin 0.0 --tmax 2.0 --output-dir event_epoch_dataset_describe
```

Potem dla każdego datasetu odpalić `train_eegnet.py` i porównać accuracy.

Cel najbliższego etapu:

```text
Znaleźć trigger i okno czasowe, które daje najlepszą klasyfikację kategorii obrazu.
```

Dopiero po tym warto iść w mocniejsze sieci, GPU/Colab, embeddingi obrazów albo próbę rekonstrukcji.

## 9. Dziennik zmian

### 2026-06-11

- Utworzono notebook dokumentacyjny `DZIENNIK_PROJEKTU_EEG.ipynb`.
- Udokumentowano aktualny pipeline przygotowania danych.
- Udokumentowano wyniki modeli: baseline, CNN, EEGNet.
- Zapisano rekomendowany następny krok: porównanie okien czasowych i triggerów dla EEGNet.

### Kolejne wpisy

Przy każdej kolejnej zmianie w projekcie będę dopisywał tutaj:

- co zostało dodane lub zmienione,
- jakie komendy zostały uruchomione,
- jaki był wynik,
- co wynika z tego dla następnego kroku.

In [ ]:
# Szybki podgląd najważniejszych plików wynikowych.
from pathlib import Path

paths = [
    Path('event_spectrogram_dataset/metadata.csv'),
    Path('event_epoch_dataset/metadata.csv'),
    Path('baseline_results/spectrogram_baseline_summary.txt'),
    Path('cnn_results/spectrogram_cnn_report.txt'),
    Path('eegnet_results/eegnet_report.txt'),
]

for path in paths:
    print(f'{path}:', 'OK' if path.exists() else 'BRAK')

## 10. Por?wnanie 5 okien czasowych dla EEGNet

**Data:** 2026-06-11  
**Sesja referencyjna:** `mole`, pliki z `dane/Wyniki`  
**Model:** EEGNet  
**Podzia?:** grupowany po `series_id`, test: serie `[2, 8]`  
**Liczba pr?bek w ka?dym wariancie:** `1980`

Por?wnano pi?? okien wok?? triggera `12 = IMAGE_ON`. Jedno okno by?o kontrolne, ca?kowicie przed pokazaniem obrazu.

| Okno | Zakres wzgl?dem IMAGE_ON | Wej?cie EEGNet | Best accuracy |
|---|---:|---:|---:|
| `image_on_0_0p8` | `0.0 .. 0.8 s` | `1 x 21 x 480` | **30.45%** |
| `image_on_m02_1p0` | `-0.2 .. 1.0 s` | `1 x 21 x 720` | 28.86% |
| `image_on_0_0p5` | `0.0 .. 0.5 s` | `1 x 21 x 300` | 27.05% |
| `image_on_m05_1p5` | `-0.5 .. 1.5 s` | `1 x 21 x 1200` | 26.36% |
| `image_on_pre_m08_0` | `-0.8 .. 0.0 s` | `1 x 21 x 480` | 16.14% |

Wniosek: najlepsze jest kr?tkie okno po pokazaniu obrazu, `0.0 .. 0.8 s`. D?ugie okno `-0.5 .. 1.5 s` prawdopodobnie dodaje szum albo mniej istotne fragmenty sygna?u. Okno przed bod?cem daje tylko `16.14%`, wi?c cz??? informacji mo?e wynika? ze struktury serii/t?a, ale g??wny sygna? ro?nie po `IMAGE_ON`.

Pliki wynikowe:

```text
event_epoch_window_grid/window_grid_manifest.csv
eegnet_window_results/window_comparison_summary.csv
eegnet_window_results/*/eegnet_report.txt
eegnet_window_results/*/eegnet.pt
```


## 11. Dodatkowe d?ugie sesje w `dane/Wyniki`

Wykryto dodatkowe nieobrobione d?ugie pliki EDF z pe?nymi danymi eksperymentu. Zosta? utworzony manifest:

```text
visual_recall_sessions_manifest.csv
```

| Uczestnik | EDF | IMAGE_ON events | Rozmiar EDF |
|---|---|---:|---:|
| `abc` | `dane/Wyniki/abc_raw.edf` | 1320 | 292.99 MB |
| `Bear` | `dane/Wyniki/Bear_raw.edf` | 2200 | 309.03 MB |
| `fghx` | `dane/Wyniki/fghx_raw.edf` | 2640 | 324.37 MB |
| `mole` | `dane/Wyniki/mole_0004_raw.edf` | 1980 | 316.73 MB |
| `Reshi` | `dane/Wyniki/Reshi_raw.edf` | 1980 | 310.64 MB |

Nast?pny wi?kszy etap powinien wykorzysta? najlepsze okno `IMAGE_ON 0.0 .. 0.8 s` i zbudowa? dataset wielosesyjny z tych pi?ciu plik?w. Wtedy test powinien by? robiony po uczestniku albo po sesji, ?eby sprawdzi? generalizacj? mi?dzy osobami.


### 2026-06-11, aktualizacja: por?wnanie okien

- Dodano `build_event_epoch_window_grid.py`.
- Dodano `discover_visual_recall_sessions.py`.
- Dodano `aggregate_eegnet_window_results.py`.
- Wygenerowano pi?? wariant?w okien czasowych wok?? `IMAGE_ON`.
- Wytrenowano EEGNet dla ka?dego okna.
- Najlepsze okno: `IMAGE_ON 0.0 .. 0.8 s`, accuracy `30.45%`.
- Wykryto pi?? d?ugich sesji w `dane/Wyniki`, gotowych do nast?pnego etapu wielosesyjnego.


## 12. Wi?kszy fine sweep okien czasowych

**Data:** 2026-06-11  
**Cel:** zag??ci? testy wok?? najlepszego obszaru po `IMAGE_ON`.  
**Model:** EEGNet  
**Podzia?:** grupowany po `series_id`, test: serie `[2, 8]`  
**Liczba pr?bek w ka?dym wariancie:** `1980`

Wygenerowano dataset fine-grid w folderze:

```text
event_epoch_window_grid_fine/
```

Wyniki zebrano w:

```text
eegnet_window_results_fine/window_comparison_summary.csv
```

| Okno | Zakres wzgl?dem IMAGE_ON | Wej?cie EEGNet | Best accuracy |
|---|---:|---:|---:|
| `image_on_0_0p8` | `0.0 .. 0.8 s` | `1 x 21 x 480` | **30.45%** |
| `image_on_0p1_0p9` | `0.1 .. 0.9 s` | `1 x 21 x 480` | **30.45%** |
| `image_on_0_0p7` | `0.0 .. 0.7 s` | `1 x 21 x 420` | 29.77% |
| `image_on_0_1p0` | `0.0 .. 1.0 s` | `1 x 21 x 600` | 29.32% |
| `image_on_0p2_0p8` | `0.2 .. 0.8 s` | `1 x 21 x 360` | 29.32% |
| `image_on_0p1_0p8` | `0.1 .. 0.8 s` | `1 x 21 x 420` | 28.64% |
| `image_on_0_0p6` | `0.0 .. 0.6 s` | `1 x 21 x 360` | 28.18% |
| `image_on_0_0p9` | `0.0 .. 0.9 s` | `1 x 21 x 540` | 26.14% |

Wniosek: wi?kszy sweep nie przebi? `30.45%`. Najbardziej stabilny wyb?r nadal wygl?da na `IMAGE_ON 0.0 .. 0.8 s`, bo jest kr?tsze i prostsze interpretacyjnie ni? przesuni?te `0.1 .. 0.9 s`, a osi?ga ten sam wynik.

Nast?pny sensowny krok to nie kolejne strojenie okien na jednej sesji, tylko dataset wielosesyjny dla okna `0.0 .. 0.8 s` i test generalizacji mi?dzy uczestnikami.


### 2026-06-11, aktualizacja: fine sweep okien

- Dodano preset `fine` w `build_event_epoch_window_grid.py`.
- Dodano `run_eegnet_window_sweep.py`.
- Wygenerowano 8 wariant?w okien czasowych w `event_epoch_window_grid_fine/`.
- Wytrenowano EEGNet dla 8 okien.
- Najlepszy wynik pozosta? `30.45%` dla `0.0 .. 0.8 s`, z remisem dla `0.1 .. 0.9 s`.
- Rekomendacja: zako?czy? strojenie okien na sesji `mole` i przej?? do walidacji wielosesyjnej.


## 13. Dataset wielosesyjny dla najlepszego okna

**Data:** 2026-06-11  
**Okno:** `IMAGE_ON 0.0 .. 0.8 s`  
**Cel:** sprawdzi? generalizacj? mi?dzy uczestnikami/sesjami.

Utworzono dataset wielosesyjny:

```text
event_epoch_multisession_image_on_0_0p8/
```

| Uczestnik | Liczba epok |
|---|---:|
| `abc` | 1320 |
| `Bear` | 2200 |
| `fghx` | 2640 |
| `mole` | 1980 |
| `Reshi` | 1980 |
| **Razem** | **10120** |

Ka?da epoka ma kszta?t:

```text
channels x samples = 21 x 480
EEGNet input = 1 x 21 x 480
```


## 14. Pierwszy test leave-one-participant

**Model:** EEGNet  
**Train:** `abc`, `Bear`, `fghx`, `Reshi`  
**Test:** `mole`  
**Epoki:** 12  
**Batch size:** 256  

Wynik:

```text
Losowo:                  9.09%
Jedna sesja mole:       30.45%
Test mi?dzy osobami:    21.67%
```

Interpretacja: wynik mi?dzy uczestnikami jest ni?szy ni? test wewn?trz jednej sesji, ale nadal wyra?nie powy?ej losowego poziomu. To oznacza, ?e cz??? sygna?u generalizuje mi?dzy osobami, ale model mocno odczuwa r??nice osobnicze / sesyjne.

Pliki wynikowe:

```text
eegnet_multisession_results/test_mole/eegnet_report.txt
eegnet_multisession_results/test_mole/eegnet.pt
```


### 2026-06-11, aktualizacja: u?ycie najlepszego okna na wielu sesjach

- Dodano `build_multi_session_epoch_dataset.py`.
- Rozszerzono `train_eegnet.py` o `--split participant` i `--test-participant`.
- Wygenerowano dataset `event_epoch_multisession_image_on_0_0p8` z 10120 epokami.
- Uruchomiono pierwszy test leave-one-participant: test na `mole`.
- Wynik EEGNet: `21.67%`, czyli powy?ej losowego `9.09%`, ale poni?ej wyniku wewn?trz jednej sesji `30.45%`.
- Nast?pny krok: uruchomi? leave-one-participant dla pozosta?ych uczestnik?w i policzy? ?redni?.


## 15. Pe?ny leave-one-participant dla 5 uczestnik?w

**Data:** 2026-06-11  
**Dataset:** `event_epoch_multisession_image_on_0_0p8`  
**Okno:** `IMAGE_ON 0.0 .. 0.8 s`  
**Model:** EEGNet  
**Epoki:** 12  
**Batch size:** 256

Uruchomiono test leave-one-participant dla ka?dego uczestnika. Wyniki s? zapisane w:

```text
eegnet_multisession_results/participant_loo_summary.csv
```

| Testowany uczestnik | Accuracy |
|---|---:|
| `mole` | 21.67% |
| `Bear` | 19.77% |
| `abc` | 19.02% |
| `fghx` | 17.42% |
| `Reshi` | 16.26% |
| **?rednia** | **18.83%** |

Por?wnanie:

```text
Losowo:                         9.09%
Najlepsze okno, jedna sesja:   30.45%
Leave-one-participant mean:    18.83%
```

Wniosek: model uczy si? sygna?u, kt?ry cz??ciowo przechodzi mi?dzy uczestnikami, ale r??nice osobnicze/sesyjne s? bardzo du?e. To jest teraz g??wny problem projektu. Dalsze kroki powinny i?? w stron? normalizacji mi?dzy uczestnikami, domain adaptation albo treningu per-participant + kalibracji.


### 2026-06-11, aktualizacja: pe?ny LOO

- Dodano `run_eegnet_participant_loo.py`.
- Dodano `aggregate_participant_loo_results.py`.
- Uruchomiono leave-one-participant dla `abc`, `Bear`, `fghx`, `mole`, `Reshi`.
- ?redni wynik EEGNet: `18.83%`.
- Najlepszy testowany uczestnik: `mole`, `21.67%`.
- Najtrudniejszy testowany uczestnik: `Reshi`, `16.26%`.


## 16. Normalizacja per uczestnik + balansowanie samplera

**Data:** 2026-06-11  
**Cel:** poprawi? generalizacj? mi?dzy uczestnikami.

Dodano do `train_eegnet.py`:

- `--normalization global|participant|epoch`,
- `--balanced-sampler none|category|participant|category_participant`.

Uruchomiony wariant:

```text
--normalization participant
--balanced-sampler category_participant
```

Wyniki leave-one-participant:

| Testowany uczestnik | Global baseline | Participant + balanced |
|---|---:|---:|
| `mole` | 21.67% | 21.57% |
| `Bear` | 19.77% | 19.05% |
| `abc` | 19.02% | 21.06% |
| `fghx` | 17.42% | 18.48% |
| `Reshi` | 16.26% | 17.42% |
| **?rednia** | **18.83%** | **19.52%** |

Wniosek: normalizacja per uczestnik z balansowaniem daje ma??, ale realn? popraw? ?redniej LOO. Poprawia trudniejsze osoby (`abc`, `fghx`, `Reshi`), ale lekko pogarsza `Bear` i praktycznie nie zmienia `mole`.

Plik wynikowy:

```text
eegnet_multisession_results_participant_balanced/participant_loo_summary.csv
```


### 2026-06-11, aktualizacja: normalizacja i balansowanie

- Rozszerzono `train_eegnet.py` o normalizacj? `participant` i `epoch`.
- Dodano `WeightedRandomSampler` z trybami balansowania.
- Rozszerzono `run_eegnet_participant_loo.py` i agregator wynik?w o nowe opcje.
- Uruchomiono pe?ny LOO dla `participant + category_participant`.
- ?rednia LOO wzros?a z `18.83%` do `19.52%`.


## 17. Poprawiona metodologia: walidacja zamiast wyboru po tescie

**Data:** 2026-06-12  
**Cel:** usunac optymizm wynikajacy z wyboru najlepszego checkpointu po wyniku testowym.

W `scripts/train_eegnet.py` dodano osobny split walidacyjny. Od tej wersji:

- trening uczy model na train split,
- checkpoint wybierany jest po validation accuracy,
- test participant / test split jest liczony dopiero raz na koncu,
- summary zapisuje `best_epoch`, `best_validation_accuracy`, `final_test_accuracy` i `test_selected_by`.

Nowy bazowy LOO:

```text
Dataset: event_epoch_multisession_image_on_0_0p8
Model: EEGNet
Split: leave-one-participant-out
Checkpoint: validation_accuracy
Epoki: 12
Batch size: 256
```

Wyniki:

| Testowany uczestnik | Accuracy |
|---|---:|
| `mole` | 21.26% |
| `Bear` | 17.32% |
| `abc` | 17.27% |
| `Reshi` | 15.15% |
| `fghx` | 14.92% |
| **Srednia** | **17.19%** |

Porownanie z wynikiem historycznym:

```text
LOO historyczne, checkpoint wybierany po tescie: 18.83%
LOO poprawione, checkpoint wybierany po walidacji: 17.19%
```

Wniosek: wynik spada po usunieciu selection bias, ale nadal zostaje wyraznie ponad poziomem losowym `9.09%`.


## 18. Kontrole negatywne

**Data:** 2026-06-12  
**Cel:** sprawdzic, czy wynik EEGNet nie wynika z przecieku etykiet, dryftu sesji albo przypadkowego kontekstu czasowego.

Dodano dwa mechanizmy kontrolne:

1. `--label-control permute` w `scripts/train_eegnet.py` - miesza etykiety `image_category` deterministycznie wedlug seeda.
2. `scripts/build_random_epoch_control_dataset.py` - buduje dataset losowych okien EEG dopasowany liczba probek i dlugoscia epoki do datasetu eventowego.

Tabela kontroli:

| Eksperyment | Test accuracy | Validation accuracy | Interpretacja |
|---|---:|---:|---|
| LOO poprawione, realne eventy | 17.19% | 16.84% | Sygna? ponad losowym poziomem. |
| LOO z permutacja etykiet | 8.93% | 9.76% | Poziom losowy. |
| LOO na losowych oknach EEG | 9.20% | 9.83% | Poziom losowy. |

Poziom losowy dla 11 klas:

```text
9.09%
```

Wniosek: kontrole negatywne dzialaja poprawnie. Aktualne wyniki nie wygladaja na prosty przeciek etykiet ani efekt przypadkowego fragmentu sesji.


## 19. Split po niewidzianych obrazach

**Data:** 2026-06-12  
**Cel:** sprawdzic, czy model generalizuje na niewidziane `image_id`, a nie tylko zapamietuje reakcje na konkretne obrazy z treningu.

Komenda:

```powershell
python .\scripts\train_eegnet.py --dataset-dir event_epoch_multisession_image_on_0_0p8 --output-dir eegnet_image_split_results --split image --epochs 12 --batch-size 256
```

Wynik:

```text
Test images: 44
Train / validation / test: 6688 / 1408 / 2024
Best validation accuracy: 23.15%
Final test accuracy:      20.80%
```

Wniosek: model pozostaje ponad losowym poziomem rowniez na niewidzianych obrazach. To sugeruje, ze nie opiera sie wylacznie na zapamietaniu konkretnych `image_id`.


## 20. Aktualnie najlepszy wariant LOO

**Data:** 2026-06-12  
**Wariant:** normalizacja per uczestnik + sampler `category_participant`  
**Cel:** sprawdzic, czy wczesniejsza mala poprawa utrzyma sie w poprawionej metodologii z walidacja.

Komenda:

```powershell
python .\scripts\run_eegnet_participant_loo.py --dataset-dir event_epoch_multisession_image_on_0_0p8 --results-parent eegnet_multisession_results_validated_participant_balanced --epochs 12 --batch-size 256 --normalization participant --balanced-sampler category_participant --force
```

Wyniki:

| Testowany uczestnik | Accuracy |
|---|---:|
| `mole` | 23.64% |
| `abc` | 16.89% |
| `fghx` | 16.40% |
| `Bear` | 16.32% |
| `Reshi` | 15.56% |
| **Srednia** | **17.76%** |

Porownanie:

```text
LOO poprawione, global normalization:       17.19%
LOO participant + category_participant:     17.76%
Roznica:                                    +0.58 pp
```

Wniosek: `--normalization participant --balanced-sampler category_participant` jest aktualnie najlepszym sprawdzonym wariantem LOO, ale poprawa jest mala. Dalszy zysk prawdopodobnie wymaga QC epok i/lub bardziej swiadomej adaptacji miedzy uczestnikami.


## 21. Aktualne pliki wynikowe i nastepne kroki

Najwazniejsze pliki wynikowe:

```text
eegnet_multisession_results_validated/participant_loo_summary.csv
eegnet_multisession_results_validated_participant_balanced/participant_loo_summary.csv
eegnet_multisession_results_permuted/participant_loo_summary.csv
eegnet_random_control_results/participant_loo_summary.csv
eegnet_image_split_results/eegnet_summary.json
event_epoch_random_control/metadata.csv
docs/WYNIKI_EKSPERYMENTOW.md
```

Aktualny ranking eksperymentow:

| Eksperyment | Wynik |
|---|---:|
| Split po `image_id` | 20.80% |
| LOO participant + balanced | 17.76% |
| LOO poprawione global | 17.19% |
| Losowe okna EEG | 9.20% |
| Permutacja etykiet | 8.93% |

Nastepne najlepsze kroki:

1. Dodac QC epok: amplitude, peak-to-peak, flatline/saturacje, liczba odrzuconych probek per uczestnik.
2. Uruchomic LOO po QC i sprawdzic, czy `participant + category_participant` nadal wygrywa.
3. Dodac split mieszany: held-out participant + held-out `image_id`.
4. Dopiero po QC rozwazac mocniejsze modele albo strojenie EEGNet.


## 22. QC epok, split participant-image i baseline bandpower

Data: 2026-06-12  
Commit: `5cd26cc Add epoch QC and bandpower baseline`

Ustalono tez zasade pracy: po kazdej zakonczonej zmianie projektowej aktualizowany jest ten notebook Jupyter jako glowny dziennik projektu.

### Co dodano

- Raportowanie QC epok w builderach datasetow epok:
  - `epoch_qc.csv` z metrykami kandydackich epok,
  - `epoch_qc_summary.csv` z podsumowaniem odrzucen i amplitud,
  - kolumny `qc_accepted`, `qc_reject_reason`, `qc_ptp_max_uv`, `qc_max_abs_uv`, `qc_flat_channel_count` w `metadata.csv`.
- Opcje `--drop-rejected` w builderach epok oraz `--only-qc-accepted` w treningu EEGNet i skryptach batchowych.
- Split `--split participant_image`, ktory testuje jednoczesnie held-out participant i held-out `image_id`.
- Nowy klasyczny baseline `scripts/train_epoch_bandpower_baseline.py` na pasmach `delta/theta/alpha/beta/gamma`.
- Raport EEGNet odporny na male testy, w ktorych nie wszystkie klasy wystepuja w `y_true`/`y_pred`.

### Dotkniete glowne pliki

- `scripts/build_event_epoch_dataset.py`
- `scripts/build_multi_session_epoch_dataset.py`
- `scripts/build_event_epoch_window_grid.py`
- `scripts/build_random_epoch_control_dataset.py`
- `scripts/train_eegnet.py`
- `scripts/train_epoch_bandpower_baseline.py`
- `scripts/run_eegnet_participant_loo.py`
- `scripts/run_eegnet_window_sweep.py`
- `docs/STRUKTURA_PROJEKTU.md`
- `.gitignore`

### Weryfikacja

- `python -m py_compile` dla zmienionych skryptow.
- Smoke datasetu QC na 8 epokach z sesji `mole`.
- Smoke trening EEGNet z `--only-qc-accepted`.
- Walidacja splitu `participant_image`: dla testowego `mole` trening nie zawieral `mole`, a overlap `image_id` wyniosl `0`.
- Smoke baseline bandpower.
- Smoke siatki okien i losowych okien kontrolnych.
- `git diff --check` bez bledow whitespace.

### Nastepny krok eksperymentalny

Zbudowac pelny dataset:

```powershell
python ./scripts/build_multi_session_epoch_dataset.py --output-dir event_epoch_multisession_image_on_0_0p8_qc --drop-rejected
```

Nastepnie uruchomic LOO z filtrem QC:

```powershell
python ./scripts/run_eegnet_participant_loo.py --dataset-dir event_epoch_multisession_image_on_0_0p8_qc --results-parent eegnet_multisession_results_qc --epochs 12 --batch-size 256 --normalization participant --balanced-sampler category_participant --only-qc-accepted --force
```


## 23. Pelny eksperyment QC epok i test participant-image

Data: 2026-06-12

### Dataset QC

Zbudowano pelny dataset:

```text
event_epoch_multisession_image_on_0_0p8_qc
IMAGE_ON, okno 0.0..0.8 s, 9014 epok po QC z 10120 kandydatow
```

Odrzucono `1106 / 10120` epok, czyli `10.93%`. Wszystkie odrzucenia byly przez prog `peak-to-peak`; nie bylo epok z `NaN/inf` ani pelnych flatline.

| Uczestnik | Epoki po QC | Kandydaci | Odrzucone | Odrzucone % |
|---|---:|---:|---:|---:|
| abc | 1159 | 1320 | 161 | 12.20% |
| Bear | 1935 | 2200 | 265 | 12.05% |
| fghx | 2336 | 2640 | 304 | 11.52% |
| mole | 1661 | 1980 | 319 | 16.11% |
| Reshi | 1923 | 1980 | 57 | 2.88% |

### LOO EEGNet po QC

Uruchomiono:

```powershell
python ./scripts/run_eegnet_participant_loo.py --dataset-dir event_epoch_multisession_image_on_0_0p8_qc --results-parent eegnet_multisession_results_qc --epochs 12 --batch-size 256 --normalization participant --balanced-sampler category_participant --only-qc-accepted --force
```

Wynik sredni LOO:

```text
mean test accuracy: 19.13%
mean validation accuracy: 18.86%
```

Wyniki per uczestnik:

| Test participant | Test accuracy | Validation accuracy |
|---|---:|---:|
| Bear | 16.95% | 22.00% |
| Reshi | 17.37% | 21.48% |
| abc | 21.48% | 17.37% |
| fghx | 16.01% | 15.55% |
| mole | 23.84% | 17.89% |

Najwazniejszy wniosek: po odrzuceniu artefaktowych epok wynik wzrosl z poprzedniego najlepszego `17.76%` do `19.13%`, wiec sygnal nie znika po QC.

### Split participant-image na QC

Uruchomiono mocniejszy test dla `mole`, gdzie trening nie zawiera ani uczestnika testowego, ani obrazow testowych:

```powershell
python ./scripts/train_eegnet.py --dataset-dir event_epoch_multisession_image_on_0_0p8_qc --output-dir eegnet_participant_image_qc_results --split participant_image --test-participant mole --epochs 12 --batch-size 256 --val-split participant --normalization participant --balanced-sampler category_participant --only-qc-accepted --cpu
```

Wynik:

```text
train/validation/test: 4336/1540/333
test images: 44
excluded samples: 2805
validation accuracy: 17.14%
test accuracy: 20.42%
```

### Baseline bandpower

Uruchomiono klasyczny baseline na pasmach mocy dla tego samego splitu:

```powershell
python ./scripts/train_epoch_bandpower_baseline.py --dataset-dir event_epoch_multisession_image_on_0_0p8_qc --output-dir baseline_epoch_bandpower_qc_results --split participant_image --test-participant mole --only-qc-accepted
```

Wynik:

```text
dummy accuracy: 4.20%
logistic regression accuracy: 11.71%
```

Interpretacja: klasyczne bandpower jest tylko lekko ponad losowe i wyraznie slabsze od EEGNet, wiec EEGNet prawdopodobnie korzysta z informacji czasowo-przestrzennej, ktorej prosty baseline pasmowy nie lapie.

### Pliki wynikowe

- `event_epoch_multisession_image_on_0_0p8_qc/session_summary.csv`
- `event_epoch_multisession_image_on_0_0p8_qc/epoch_qc_summary.csv`
- `eegnet_multisession_results_qc/participant_loo_summary.csv`
- `eegnet_participant_image_qc_results/eegnet_summary.json`
- `baseline_epoch_bandpower_qc_results/bandpower_baseline_summary.json`

### Nastepne kroki

- Powtorzyc split `participant_image` dla kazdego uczestnika, nie tylko dla `mole`.
- Dodac kontrole negatywne dla datasetu QC: permutacja etykiet i losowe okna po takim samym QC.
- Sprawdzic stabilnosc progow QC, np. `150 uV` vs `200 uV`.


## 24. Pelny sweep participant-image na datasetcie QC

Data: 2026-06-12

Dodano skrypt:

```text
scripts/run_eegnet_participant_image.py
```

Skrypt uruchamia EEGNet dla splitu `participant_image` po wszystkich uczestnikach, czyli testuje jednoczesnie held-out participant oraz held-out `image_id`.

### Komenda

```powershell
python ./scripts/run_eegnet_participant_image.py --dataset-dir event_epoch_multisession_image_on_0_0p8_qc --results-parent eegnet_participant_image_qc_sweep_results --epochs 12 --batch-size 256 --only-qc-accepted --cpu --force
```

Ustawienia:

```text
normalization: participant
balanced_sampler: category_participant
validation split: participant
checkpoint selected by: validation_accuracy
```

### Wyniki

| Test participant | Test samples | Test accuracy | Validation accuracy | Best epoch |
|---|---:|---:|---:|---:|
| Bear | 392 | 14.80% | 23.62% | 10 |
| Reshi | 383 | 16.71% | 22.65% | 10 |
| abc | 232 | 17.24% | 17.86% | 9 |
| fghx | 470 | 15.53% | 15.71% | 9 |
| mole | 333 | 20.42% | 17.14% | 5 |
| **MEAN** | 1810 lacznie | **16.94%** | **19.40%** | n/a |

Dodatkowo srednia wazona po liczbie probek testowych wyniosla `16.74%`.

### Interpretacja

Pelny `participant_image` jest trudniejszy niz LOO QC, bo usuwa z treningu zarowno uczestnika testowego, jak i obrazy testowe. Wynik `16.94%` jest nizszy niz LOO QC `19.13%`, ale nadal wyraznie ponad poziom losowy `9.09%` dla 11 klas.

Wczesniejszy pojedynczy run dla `mole` (`20.42%`) zostal odtworzony w pelnym sweepie.

### Pliki wynikowe

- `eegnet_participant_image_qc_sweep_results/participant_image_summary.csv`
- `eegnet_participant_image_qc_sweep_results/test_Bear/eegnet_summary.json`
- `eegnet_participant_image_qc_sweep_results/test_Reshi/eegnet_summary.json`
- `eegnet_participant_image_qc_sweep_results/test_abc/eegnet_summary.json`
- `eegnet_participant_image_qc_sweep_results/test_fghx/eegnet_summary.json`
- `eegnet_participant_image_qc_sweep_results/test_mole/eegnet_summary.json`

### Nastepne kroki

- Uruchomic kontrole negatywne na datasetcie QC: permutacja etykiet i losowe okna po QC.
- Uruchomic baseline bandpower dla pelnego sweepu `participant_image`, zeby miec porownanie per uczestnik.
- Sprawdzic stabilnosc progow QC, np. `150 uV` vs `200 uV`.
